# Orbital symmetry blocks and the winding number

When the active orbitals fall into irreducible representations that do not mix, $G$ is
block diagonal and $\det G$ factorises. Each block then carries its own **winding number**:
the number of times $\det G(z)$ circles the origin as $z$ traverses a closed contour. By the
argument principle that integer is

$$
\text{winding} = (\text{zeros of } \det G \text{ inside}) - (\text{poles inside})
$$

and an integer cannot change continuously.

Worked on cyclobutadiene, whose automerization — the two double bonds trading places
through a square transition state — is the textbook orbital-symmetry-controlled process.
The rectangle is a clean D2h case. The square is the interesting one, and this notebook
ends there: it is where symmetry blocking stops being well defined at all, for a reason
worth understanding.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import cm
from pyscf import gto, symm

from casgf import (
    ActiveSpace,
    block_leakage,
    det_along_contour,
    irrep_blocks,
    keyhole_contour,
    lehmann,
    min_pole_distance,
    parity_blocks,
    run_casscf,
    winding_number,
    winding_number_of,
)

CONTOUR = keyhole_contour(radius=1.5, n_per_segment=10_000)

In [ ]:
CH = 1.08          # C-H bond length, Angstrom
PERIMETER = 2.90   # a + b held fixed along the scan


def cyclobutadiene(a, b=None):
    """Planar cyclobutadiene with C-C bonds of alternating length `a` and `b`.

    Carbons sit on the corners of a rectangle; each C-H bond runs along the
    exterior angle bisector, which is the ring diagonal. `a == b` is the square
    D4h transition state of the automerization; the D2h ground state has
    alternating short and long bonds.

    Idealised, not optimised -- good enough to show the physics, and it keeps
    the notebook self-contained.
    """
    b = PERIMETER - a if b is None else b
    atoms = []
    for sx, sy in ((1, 1), (-1, 1), (-1, -1), (1, -1)):
        cx, cy = sx * a / 2, sy * b / 2
        atoms.append(f"C {cx:.8f} {cy:.8f} 0.0")
        atoms.append(f"H {cx + sx * CH / np.sqrt(2):.8f} {cy + sy * CH / np.sqrt(2):.8f} 0.0")
    return "; ".join(atoms)


def benzene(cc=1.39, ch=1.09):
    """Idealised D6h benzene: a regular hexagon of carbons with radial C-H bonds."""
    atoms = []
    for k in range(6):
        c, s = np.cos(2 * np.pi * k / 6), np.sin(2 * np.pi * k / 6)
        atoms.append(f"C {cc * c:.8f} {cc * s:.8f} 0.0")
        atoms.append(f"H {(cc + ch) * c:.8f} {(cc + ch) * s:.8f} 0.0")
    return "; ".join(atoms)

## Running CASSCF, then labelling the orbitals

Note `symmetry=False` in the CASSCF itself. Constraining the orbitals to irreps of the
abelian subgroup makes the solver fail at the square, where the frontier pair is degenerate
— so the calculation is run unconstrained and the converged orbitals are classified
afterwards. `symm.label_orb_symm` refuses if they are not symmetry adapted, which turns out
to be the informative case here.

In [ ]:
def analyse(a, basis="def2-SVP"):
    """CASSCF(4,4) on cyclobutadiene, plus symmetry labels for the active orbitals."""
    atom = cyclobutadiene(a)
    mol = gto.M(atom=atom, basis=basis, unit="A", verbose=0)
    mc = run_casscf(mol, ncas=4, nelecas=4)
    space = ActiveSpace.from_casscf(mc)

    symmetric = gto.M(atom=atom, basis=basis, unit="A", symmetry=True, verbose=0)
    active = slice(mc.ncore, mc.ncore + mc.ncas)
    try:
        space.orbsym = np.asarray(symm.label_orb_symm(
            symmetric, symmetric.irrep_name, symmetric.symm_orb,
            np.asarray(mc.mo_coeff)[:, active],
        ))
        note = f"orbital symmetries: {', '.join(map(str, space.orbsym))}"
    except ValueError as err:
        space.orbsym = None
        note = f"labelling refused -- {err}"

    print(f"a = {a:.3f} A   point group {symmetric.topgroup}   E = {mc.e_tot:.8f} Ha")
    print(f"  {note}")
    return space, lehmann(space)

## The rectangle: clean blocking

Each of the four $\pi$ orbitals lands in its own irrep, so `G` splits into four $1\times1$
blocks. `block_leakage` measures how far that is from exactly true.

In [ ]:
space, gf = analyse(1.34)
blocks = irrep_blocks(space.orbsym)

print(f"\nblocks       { {k: v.tolist() for k, v in blocks.items()} }")
print(f"leakage      {block_leakage(gf.at(0.0, eta=0.05), blocks.values()):.1e}")
print(f"mu {gf.mu:+.6f}   gap {gf.gap:.6f}   sum rule {gf.sum_rule():.9f}")

# det G must factorise over the blocks, exactly.
freqs = np.linspace(-1, 1, 201)
product = np.ones(freqs.size, dtype=complex)
for idx in blocks.values():
    product *= gf.det(freqs, eta=0.05, orbitals=idx)
print(f"max |det G - prod(det G_block)| = "
      f"{np.abs(gf.det(freqs, eta=0.05) - product).max():.1e}")

In [ ]:
print(f"closest pole to the contour: {min_pole_distance(gf, CONTOUR):.2e}\n")

per_block = {name: winding_number_of(gf, CONTOUR, orbitals=idx) for name, idx in blocks.items()}
for name, value in per_block.items():
    print(f"  {name:4s} {value:+d}")
print(f"  {'sum':4s} {sum(per_block.values()):+d}"
      f"    (whole active space: {winding_number_of(gf, CONTOUR):+d})")

In [ ]:
def plot_traces(gf, blocks, contour, title):
    names = list(blocks)
    fig, axes = plt.subplots(1, len(names), figsize=(3.1 * len(names), 3.3), dpi=120)
    colour = np.linspace(0, 1, len(contour))
    for ax, name in zip(np.atleast_1d(axes), names, strict=True):
        values = det_along_contour(gf, contour, orbitals=blocks[name])
        ax.scatter(values.real, values.imag, s=0.2, c=colour, cmap=cm.gnuplot)
        ax.axhline(0, color="k", lw=0.5, ls="--")
        ax.axvline(0, color="k", lw=0.5, ls="--")
        ax.set_title(f"{name}   ({winding_number(values):+d})", fontsize=10)
        ax.set_xlabel(r"$\mathrm{Re}\,\det G$")
    np.atleast_1d(axes)[0].set_ylabel(r"$\mathrm{Im}\,\det G$")
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


plot_traces(gf, blocks, CONTOUR, "Cyclobutadiene rectangle — colour runs along the contour")

## Where the zeros show up

For a non-interacting Green's function $\det G$ is a product of $1/(z - \varepsilon_k)$ and
cannot vanish, so its winding number counts poles alone. Comparing the two on the same
contour isolates the contribution of the zeros.

Only poles carrying spectral weight count: most of the $N\pm1$ spectrum is orthogonal to
$c^{\dagger}|0\rangle$ and never appears in $G$.

In [ ]:
free = lehmann(ActiveSpace.from_arrays(space.h1, np.zeros_like(space.eri), space.nelecas))

print(f"{'':16s} {'winding':>8} {'poles in':>9} {'zeros in':>9}")
for label, g in (("interacting", gf), ("non-interacting", free)):
    weights = g.pole_weights()
    poles_in = int(np.count_nonzero((g.poles > 0) & (g.poles < 1.5) & (weights > 1e-10)))
    winding = winding_number_of(g, CONTOUR)
    print(f"{label:16s} {winding:+8d} {poles_in:9d} {winding + poles_in:9d}")

The non-interacting row has **exactly zero** enclosed zeros, as it must. Every zero counted
in the interacting row is produced by the interaction — and those are precisely the features
that are invisible in the spectral function.

## The square: where the blocking stops being defined

At $a = b$ the molecule is D4h and the two frontier $\pi$ orbitals form a degenerate $e_g$
pair. Any linear combination of a degenerate pair is an equally valid solution, so an
unconstrained CASSCF returns an arbitrary mixture of the two — and asking which irrep each
one belongs to has no answer.

In [ ]:
square_space, square_gf = analyse(1.45)

`label_orb_symm` reports a norm of 0.5, i.e. a 50/50 mixture, and refuses. That is the
right answer rather than a failure: **the symmetry blocking is genuinely undefined here**,
and a tool that silently returned some labelling would be hiding it.

Two things still work:

- **`block_leakage`** on any candidate blocking, as a quantitative statement about how far
  `det G` is from factorising;
- **the total winding number**, which needs no blocking at all and is basis independent.

In [ ]:
leak = block_leakage(square_gf.at(0.0, eta=0.05), parity_blocks(4).values())
print(f"square:    leakage (index parity) {leak:.1e}   "
      f"gap {square_gf.gap:.6f}   total winding {winding_number_of(square_gf, CONTOUR):+d}")
print(f"rectangle: leakage (index parity) "
      f"{block_leakage(gf.at(0.0, eta=0.05), parity_blocks(4).values()):.1e}   "
      f"gap {gf.gap:.6f}   total winding {winding_number_of(gf, CONTOUR):+d}")

## Notes

- The origin lies *on* the contour, on its imaginary-axis segment. That is harmless here:
  the chemical potential puts $\omega = 0$ in the middle of the gap, where $\det G$ is finite
  and non-zero. `winding_number` raises if $\det G$ ever vanishes exactly on the contour.
- `parity_blocks` — index parity, `G[::2, ::2]` and `G[1::2, 1::2]` — is a fallback for when
  point-group symmetry is unavailable. It is only as good as the assumption that the
  orbitals alternate between two irreps, and `block_leakage` is how you find out whether
  that holds.
- [`03_green_function_map.py`](03_green_function_map.py) runs the whole automerization path
  and plots $\log|\det G|$ across it.